## Post Generation Autonomus workflow with Iterative Approach - HUMAN IN THE LOOP

In [69]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field
from typing import Literal,List, Annotated, Optional
from dotenv import load_dotenv
import operator
import os

load_dotenv()  # Load environment variables from .env file

True

In [70]:
model = ChatGoogleGenerativeAI(api_key=os.getenv("GOOGLE_API_KEY"), model="gemini-2.5-flash-lite", temperature=0)


In [71]:
# structured output schema
class PostEvaluationSchema(BaseModel):
   evaluated_post: Literal["approved", "not_approved"] = Field(..., description="Evaluation of the generated post")
   feedback: str = Field(..., description="Feedback for improving the post if not approved")


structured_model = model.with_structured_output(PostEvaluationSchema)

In [74]:
class GenerationState(BaseModel):
    topic: str = Field(description="The main topic for content generation")

    post: Optional[str] = None
    evaluated_post: Optional[Literal["approved", "not_approved"]] = None
    feedback: Optional[str] = None

    feedback_history: Annotated[List[str], operator.add] = Field(default_factory=list)
    post_history: Annotated[List[str], operator.add] = Field(default_factory=list)

    iteration: int = Field(default=0, description="Current iteration number")
    max_iterations: int = Field(default=3, description="Maximum number of iterations allowed")
    

#### Generation Function

In [73]:
# functions for each node state
def generate_post(state: GenerationState) -> dict:

    messages = [
        SystemMessage(content="You are a funny and clever social media influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state.topic}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day English
""")
    ]

    post = model.invoke(messages).content
    return {"post": post, "post_history": [post]}



#### Evaluation Functions 

In [75]:
def evaluate_post(state: GenerationState) -> dict:
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given social media critic. You evaluate tweets/posts based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet/post:

Tweet: "{state.post}"

Use the criteria below to evaluate the tweet/post:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluated_post: "approved" or "not_approved"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    evaluation = structured_model.invoke(messages)

    return {
        "evaluated_post": evaluation.evaluated_post,
        "feedback": evaluation.feedback,
        'feedback_history': [evaluation.feedback],
    }



#### Optimizer Function 

In [76]:
def optimize_post(state: GenerationState) -> dict:

    # prompt to optimize the post based on feedback
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state.feedback}"

Topic: "{state.topic}"
Original Tweet:
{state.post}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]


    optimized_post = model.invoke(messages).content

    return {
        "post": optimized_post,
        "post_history": [optimized_post],
        "iteration": state.iteration + 1
    }

#### Check Approved / Not Condition

In [77]:
def check_condition(state: GenerationState) -> str:
    if state.evaluated_post == "approved":
        return "approved"
    elif state.iteration >= state.max_iterations:
        return "approved"
    else:
        return "not_approved"

In [78]:
graph = StateGraph(GenerationState)

# add modes
graph.add_node('generate_post',generate_post)
graph.add_node('evaluate_post',evaluate_post)
graph.add_node('optimize_post',optimize_post)


# add edges
graph.add_edge(START, 'generate_post')
graph.add_edge('generate_post', 'evaluate_post')
graph.add_conditional_edges('evaluate_post', check_condition,{'approved': END, 'not_approved': 'optimize_post'})
graph.add_edge('optimize_post', 'evaluate_post')

workflow = graph.compile()

In [67]:

initial_state = {"topic": "Mango Season","iteration":1,"max_iterations":5}

final_state = workflow.invoke(initial_state)

In [68]:
final_state

{'topic': 'Mango Season',
 'post': 'Mango season is here and my kitchen looks like a crime scene. Sticky fingers, yellow stains everywhere, and a faint, sweet scent of pure chaos. Worth it. 🥭😂 #MangoSeason #FruitFrenzy #SendWipes',
 'evaluated_post': 'approved',
 'feedback': "This tweet is approved. It's relatable and humorous, painting a vivid picture of the delicious mess that mango season brings. The use of emojis and relevant hashtags enhances its shareability. The length is perfect for Twitter, and the 'worth it' ending provides a satisfying conclusion without being deflating. It's a solid, engaging tweet.",
 'feedback_history': ["This tweet is approved. It's relatable and humorous, painting a vivid picture of the delicious mess that mango season brings. The use of emojis and relevant hashtags enhances its shareability. The length is perfect for Twitter, and the 'worth it' ending provides a satisfying conclusion without being deflating. It's a solid, engaging tweet."],
 'post_hist